Import the required libraries

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score, recall_score, precision_score
#SimpleImputer ka use missing values (NaN) ko fill karne ke liye hota hai.
from sklearn.impute import SimpleImputer
#ColumnTransformer
from sklearn.compose import ColumnTransformer



Load the data


In [4]:
diabetic_data = pd.read_csv('../dataset/diabetes.csv')
diabetic_data.head(15)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


Data Understand

In [5]:
diabetic_data.shape

(768, 9)

In [6]:
diabetic_data.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

columns

In [7]:
diabetic_data.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [8]:
diabetic_data['Outcome'].value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

Replace all the invalid zeros with NaN

In [9]:
cols_with_invalid_zeros = [
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI'
]

for col in cols_with_invalid_zeros:
  diabetic_data[col] = diabetic_data[col].replace(0, np.nan)

Define the features and target

In [10]:
X = diabetic_data.drop('Outcome', axis=1)
y = diabetic_data['Outcome']

Train-Test Split


In [11]:
# stratify => used for classification tasks especially when the data is highly imbalanced
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [12]:
diabetic_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   763 non-null    float64
 2   BloodPressure             733 non-null    float64
 3   SkinThickness             541 non-null    float64
 4   Insulin                   394 non-null    float64
 5   BMI                       757 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(6), int64(3)
memory usage: 54.1 KB


Numerical Pipeline

In [15]:
numeric_features = X.columns.to_list()

numeric_pipeline = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy='median')),
        ("scaler", StandardScaler())
    ]
)

In [16]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_pipeline, numeric_features)
    ]
)

Model Building & Training

In [17]:
model_pipeline = Pipeline(
    steps = [
        ('preprocessing', preprocessor),
        ('model', LogisticRegression())
    ]
)

In [18]:
model_pipeline.fit(X_train, y_train)


,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Model Predictions

In [19]:
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

In [20]:
y_pred


array([1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0])

In [21]:
y_prob

array([0.60969998, 0.11829145, 0.29342565, 0.25080052, 0.03352696,
       0.16847753, 0.48806448, 0.92366487, 0.0814217 , 0.83859286,
       0.2810194 , 0.47968596, 0.12920519, 0.10176183, 0.28850298,
       0.35223503, 0.72322529, 0.08447883, 0.8132411 , 0.10882619,
       0.14801945, 0.71479316, 0.17369068, 0.92856488, 0.54995719,
       0.01217342, 0.6347534 , 0.04313993, 0.31189285, 0.03711449,
       0.04559736, 0.0393708 , 0.47667962, 0.64525685, 0.88412872,
       0.13735601, 0.35110159, 0.06002218, 0.78813911, 0.61799382,
       0.29338912, 0.09868878, 0.0878967 , 0.27793039, 0.14647696,
       0.4126086 , 0.1331809 , 0.10276637, 0.68520735, 0.34500531,
       0.62682907, 0.70867639, 0.44572848, 0.06195468, 0.51091021,
       0.36327438, 0.75153351, 0.21516872, 0.77652364, 0.14405244,
       0.81517859, 0.2225239 , 0.04927818, 0.89479032, 0.03379495,
       0.1608703 , 0.89767166, 0.01616709, 0.26482376, 0.67382033,
       0.18409037, 0.09857591, 0.35284865, 0.41185601, 0.03403

Model Evaluation

In [22]:
accuracy_score(y_test, y_pred)

0.7077922077922078

In [23]:
recall_score(y_test, y_pred)

0.5

In [24]:
precision_score(y_test, y_pred)

0.6

In [25]:
confusion_matrix(y_test, y_pred)

array([[82, 18],
       [27, 27]])

In [26]:
roc_auc_score(y_test, y_prob)

0.812962962962963

Save this model

In [27]:
import joblib
joblib.dump(model_pipeline, "diabetes_pipeline.pkl")

['diabetes_pipeline.pkl']

Load the model

In [28]:
loaded_model = joblib.load('diabetes_pipeline.pkl')
y_pred_loaded_model = loaded_model.predict(X_test)
y_pred_loaded_model

array([1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0])

Customize the threshold values

In [30]:
custom_threshold = 0.3
y_custom_pred = (y_prob >= custom_threshold).astype(int)

recall_score(y_test, y_custom_pred)

0.7962962962962963

Data Imbalance

In [31]:
y_train.value_counts()

Outcome
0    400
1    214
Name: count, dtype: int64

In [32]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler()
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

y_train_over.value_counts()

Outcome
0    400
1    400
Name: count, dtype: int64